# Badge 3: Data Application Builders Workshop: Lesson 8: Intro to Variables and Variable-Driven Loading

Local variables, the SUM_MYSTERY_BAG_VARS UDF and NEUTRALIZE_WHINING.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE3_DABW_ADMIN` (ACCOUNTADMIN), `BADGE3_DABW_ROLE` (SYSADMIN) and `BADGE3_DABW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE3_DABW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE3_DABW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE3_DABW_WH"      # stands in for the course's COMPUTE_WH
import ast, io
from snowflake.snowpark.context import get_active_session

CUSTOM = 'SMOOTHIES.PUBLIC."Custom Smoothie Order Form"'
PENDING = 'SMOOTHIES.PUBLIC."Pending Smoothie Orders"'


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result


def write_app(app, app_code):
    """SQL equivalent of editing the app in Snowsight's Streamlit editor: write streamlit_app.py into the app's live version."""
    ast.parse(app_code)  # the code must at least be valid Python
    path = f"snow://streamlit/{app}/versions/live/streamlit_app.py"
    get_active_session().file.put_stream(io.BytesIO(app_code.encode()), path, auto_compress=False, overwrite=True)
    print(f"wrote {len(app_code.splitlines())} lines to {path}")

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: 💬 Zena Checks In on Mel

Story only.

## Page: ❓ A Mystery Bag Called a Variable

> **Course:** Create & Set a Local SQL Variable (screenshot 2026011615014); Run a Select that Displays the Variable (015); Change the Value and Run the Select Again (015.jpg); Do More With More Variables (020).

In [ ]:
%%sql -r set_mystery_bag
set mystery_bag = 'What is in here?';

In [ ]:
%%sql -r select_mystery_bag
select $mystery_bag;

In [ ]:
%%sql -r change_mystery_bag
set mystery_bag = 'This bag is empty!!';

select $mystery_bag;

In [ ]:
%%sql -r more_variables
set var1 = 2;
set var2 = 5;
set var3 = 7;

select $var1+$var2+$var3;

## Page: 📁 A Code Organizer Called a Function

> **Course:** Create a Simple User Defined Function (UDF). Put your function in your UTIL_DB database! (screenshot 2026011615075.png).

In [ ]:
%%sql -r create_sum_function
-- From screenshot 2026011615075.png, created in UTIL_DB.PUBLIC as the SYSADMIN stand-in
use role {{SYSADMIN_ROLE}};
use schema UTIL_DB.PUBLIC;
create function sum_mystery_bag_vars (var1 number, var2 number, var3 number)
 returns number as 'select var1+var2+var3';

> **Course:** Run Your New Function (screenshot 2026011615075.jpg).

In [ ]:
%%sql -r run_sum_function
select sum_mystery_bag_vars (12,36,204);

> **Course:** Combine Local Variables & Function Calls (screenshot 2026011615084.jpg).

In [ ]:
%%sql -r combine_variables
set eeny = 4;
set meeny = 67.2;
set miney_mo = -39;

select sum_mystery_bag_vars ($eeny,$meeny,$miney_mo);

## Page: 🤖 DORA DABW006

> **Course:** This, That & The Other: set the three local variables to -10.5, 2 and 1000, in the same session as the check.

In [ ]:
%%sql -r dabw006_variables
-- The values the course asks for. The DORA check (dora/dora_tests) sets them again in its own session.
set this = -10.5;
set that = 2;
set the_other = 1000;

In [ ]:
%%sql -r dabw006_function_check
select util_db.public.sum_mystery_bag_vars($this,$that,$the_other);

### 🤖 DORA check DABW006 goes here
The course has the learner run **DABW006** (the function sums the three local variables to 991.5) at this point.
It is in `dora/dora_tests.ipynb`, section **DABW006**, as a local check plus the exact DORA code.

## Page: ⛙ Combining UDFs with Snowflake Functions

> **Course:** Using a System Function to Fix a Variable Value (screenshot 2026011615201.jpg): set a phrase with alternating caps and use INITCAP().

In [ ]:
%%sql -r alternating_caps
set alternating_caps_phrase = 'aLtErNaTiNg CaPs!';

select $alternating_caps_phrase;

In [ ]:
%%sql -r initcap_example
set alternating_caps_phrase = 'wHy ArE yOu lIkE tHiS?';

select initcap($alternating_caps_phrase);

> **Course:** 🎯 CHALLENGE LAB: Write a UDF that Neutralizes Alternating Caps Phrases: in UTIL_DB.PUBLIC, named NEUTRALIZE_WHINING, one TEXT argument, returns TEXT, first letter of each word capitalized and the rest lower case (use INITCAP).

In [ ]:
%%sql -r create_neutralize_whining
-- Derived solution (the course gives the requirements, not the code)
use role {{SYSADMIN_ROLE}};
use schema UTIL_DB.PUBLIC;
create function neutralize_whining(phrase text)
 returns text as 'select initcap(phrase)';

In [ ]:
%%sql -r test_neutralize_whining
select neutralize_whining('bUt mOm i wAsHeD tHe dIsHes yEsTeRdAy');

## Page: 🤖 DORA DABW007

### 🤖 DORA check DABW007 goes here
The course has the learner run **DABW007** (NEUTRALIZE_WHINING works) at this point.
It is in `dora/dora_tests.ipynb`, section **DABW007**, as a local check plus the exact DORA code.